In [ ]:
"""
📁 你的專案資料夾/
├── 📄 script.py       <-- 你的 Python 程式碼
└── 📁 my_images/            <-- IMAGE_FOLDER_NAME 對應的資料夾
    ├── 001.webp
    ├── 002.jpg
    └── 003.png
"""

In [9]:
pip install Pillow

Note: you may need to restart the kernel to use updated packages.


In [11]:
import re
from pathlib import Path
from PIL import Image

def natural_sort_key(path: Path):
    """自然排序：將檔名數字轉為整數排序 (例如 001, 002... 240)"""
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', path.name)]

def convert_images_to_pdf(folder_path: Path, output_pdf_name: str = "merged_result.pdf"):
    if not folder_path.exists() or not folder_path.is_dir():
        print(f"錯誤：找不到資料夾喵！\n查找路徑：{folder_path}")
        return

    valid_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    
    # 搜尋圖片並依檔名數字自然排序
    image_paths = [
        p for p in folder_path.iterdir()
        if p.is_file() and p.suffix.lower() in valid_extensions
    ]
    image_paths.sort(key=natural_sort_key)

    if not image_paths:
        print(f"在 {folder_path} 中沒有找到符合格式的圖片喵！")
        return

    print(f"共找到 {len(image_paths)} 張圖片，開始合成 PDF 喵...")

    rgb_images = []
    for p in image_paths:
        with Image.open(p) as img:
            # 處理透明通道（RGBA、LA 或透明調色盤），填入白底避免黑塊或報錯
            if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
                rgba = img.convert("RGBA")
                background = Image.new("RGB", rgba.size, (255, 255, 255))
                background.paste(rgba, mask=rgba.split()[-1])
                rgb_images.append(background)
            else:
                rgb_images.append(img.convert("RGB"))

    output_path = folder_path / output_pdf_name
    first_image = rgb_images[0]
    first_image.save(
        output_path,
        save_all=True,
        append_images=rgb_images[1:],
        quality=95,
    )

    print(f"大功告成喵！已成功輸出至：{output_path}")

# Jupyter Notebook 取得所在資料夾請用 Path.cwd() 喵
BASE_DIR = Path.cwd()
IMAGE_FOLDER_NAME = "my_images"

target_path = BASE_DIR / IMAGE_FOLDER_NAME
convert_images_to_pdf(target_path, output_pdf_name="output.pdf")

共找到 242 張圖片，開始合成 PDF 喵...
大功告成喵！已成功輸出至：C:\Users\chenk\Downloads\完全指南\my_images\output.pdf
